# Homework 22: Causal and Multi-Head Attention

**Audience.** Students who can trace the unmasked attention calculation from Homework 21.

**Prerequisites.** Queries, keys, values, softmax attention weights, and three-dimensional batch/sequence/channel tensors.

**Learning goals.** By the end, you will be able to:

- explain why autoregressive generation requires a causal mask;
- split channels into multiple attention heads and recombine them;
- verify that changing a future suffix cannot affect earlier outputs;
- explain why positional embeddings are necessary.


## Outline

1. Prevent information from leaking out of the future
2. Implement causal multi-head attention
3. Test prefix invariance
4. Add position information
5. Notebook checkpoints


In [1]:
# S1: Imports and reproducibility
import math
import torch
from torch import nn

_ = torch.manual_seed(158)
torch.set_printoptions(precision=3, sci_mode=False)


## 1. The model must not see the answer

During next-token training, all positions are processed in parallel. The output at input position `i` predicts the token stored at input position `i+1`. Without a mask, position `i` could attend directly to position `i+1`—its target—and leak the answer into the computation. Attending to position `i` itself is allowed.

A lower-triangular mask allows row `i` to attend only to columns `0` through `i`.


In [2]:
# S2: Apply a causal mask before softmax
raw_scores = torch.tensor([
    [2.0, 1.0, 0.0, -1.0],
    [0.0, 2.0, 1.0, 0.0],
    [1.0, 0.0, 2.0, 1.0],
    [0.0, 1.0, 0.0, 2.0],
])

causal_mask = torch.tril(torch.ones(4, 4, dtype=torch.bool))
masked_scores = raw_scores.masked_fill(~causal_mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)

print("mask:\n", causal_mask.int())
print("masked scores:\n", masked_scores)
print("causal weights:\n", causal_weights)
print("row sums:", causal_weights.sum(dim=-1))
assert torch.count_nonzero(torch.triu(causal_weights, diagonal=1)) == 0


mask:
 tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
masked scores:
 tensor([[2., -inf, -inf, -inf],
        [0., 2., -inf, -inf],
        [1., 0., 2., -inf],
        [0., 1., 0., 2.]])
causal weights:
 tensor([[1.000, 0.000, 0.000, 0.000],
        [0.119, 0.881, 0.000, 0.000],
        [0.245, 0.090, 0.665, 0.000],
        [0.083, 0.225, 0.083, 0.610]])
row sums: tensor([1.000, 1.000, 1.000, 1.000])


Notice that the first row is exactly `[1, 0, 0, 0]`: the first position has no earlier tokens to retrieve. By the fourth row, all four positions are available.


## 2. Multiple heads

One attention head produces one retrieval pattern. Multiple heads let the model perform several kinds of retrieval simultaneously. We project to all queries, keys, and values in one linear layer, then reshape the channel dimension into `number_of_heads × head_size`.

`register_buffer` stores the fixed mask with the module without treating it as a trainable parameter. A registered buffer is included in the module's `state_dict` and moves automatically when the module is sent to a CPU, GPU, or other device, but it is absent from `parameters()` and is not updated by the optimizer. A plain tensor attribute would not move automatically with the module. After transposing the heads, `contiguous()` arranges the values in memory so `view` can safely combine the head dimensions again.


In [3]:
# S3: Causal multi-head self-attention
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, number_of_heads, maximum_context):
        super().__init__()
        assert d_model % number_of_heads == 0
        self.number_of_heads = number_of_heads
        self.head_size = d_model // number_of_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.projection = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(maximum_context, maximum_context, dtype=torch.bool))
        self.register_buffer("causal_mask", mask.view(1, 1, maximum_context, maximum_context))

    def forward(self, x, return_weights=False):
        batch_size, sequence_length, d_model = x.shape

        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(batch_size, sequence_length, self.number_of_heads, self.head_size).transpose(1, 2)
        k = k.view(batch_size, sequence_length, self.number_of_heads, self.head_size).transpose(1, 2)
        v = v.view(batch_size, sequence_length, self.number_of_heads, self.head_size).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_size)
        mask = self.causal_mask[:, :, :sequence_length, :sequence_length]
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)

        attended = weights @ v
        attended = attended.transpose(1, 2).contiguous()
        attended = attended.view(batch_size, sequence_length, d_model)
        output = self.projection(attended)

        if return_weights:
            return output, weights
        return output


In [4]:
# S4: Trace the multi-head shapes
attention = CausalSelfAttention(
    d_model=8,
    number_of_heads=2,
    maximum_context=10,
)
example = torch.randn(3, 6, 8)
attention_output, attention_weights = attention(example, return_weights=True)

print("input:", tuple(example.shape))
print("attention weights:", tuple(attention_weights.shape))
print("output:", tuple(attention_output.shape))
print("one weight row:", attention_weights[0, 0, 2])


input: (3, 6, 8)
attention weights: (3, 2, 6, 6)
output: (3, 6, 8)
one weight row: tensor([0.301, 0.217, 0.482, 0.000, 0.000, 0.000], grad_fn=<SelectBackward0>)


The weight tensor has shape `(batch, heads, queries, keys)`. The output returns to `(batch, sequence, d_model)` after the head outputs are concatenated and projected.


## 3. Prefix invariance

A causal layer must give exactly the same outputs for a shared prefix, even when the unseen suffix changes. This is a stronger and more useful test than merely looking at the mask.


In [5]:
# S5: Changing positions 3-5 cannot change outputs at positions 0-2
first_sequence = torch.randn(1, 6, 8)
second_sequence = first_sequence.clone()
second_sequence[:, 3:] = torch.randn(1, 3, 8)

first_output = attention(first_sequence)
second_output = attention(second_sequence)

prefix_difference = (first_output[:, :3] - second_output[:, :3]).abs().max()
suffix_difference = (first_output[:, 3:] - second_output[:, 3:]).abs().max()

print("largest prefix difference:", float(prefix_difference.detach()))
print("largest suffix difference:", float(suffix_difference.detach()))
assert torch.allclose(first_output[:, :3], second_output[:, :3], atol=1e-6)


largest prefix difference: 0.0
largest suffix difference: 0.3545757830142975


## 4. Explicit position information

Unmasked self-attention without position information is permutation-equivariant: reordering its inputs simply reorders its outputs. A causal mask already introduces prefix asymmetry, so it is too strong to say that a causal model has no order information at all. We nevertheless add learned position vectors because they give the model direct access to absolute position and make distances and order much easier to represent.


In [6]:
# S6: Identical tokens become different inputs at different positions
token_embedding = nn.Embedding(20, 8)
position_embedding = nn.Embedding(10, 8)

repeated_token_ids = torch.tensor([[7, 7, 7, 7]])
positions = torch.arange(repeated_token_ids.shape[1])

token_vectors = token_embedding(repeated_token_ids)
transformer_inputs = token_vectors + position_embedding(positions)

print("token vectors at positions 0 and 1 equal:",
      torch.allclose(token_vectors[0, 0], token_vectors[0, 1]))
print("combined inputs at positions 0 and 1 equal:",
      torch.allclose(transformer_inputs[0, 0], transformer_inputs[0, 1]))


token vectors at positions 0 and 1 equal: True
combined inputs at positions 0 and 1 equal: False


## Notebook checkpoints

Predict the mask row, tensor shapes, and invariance result before running this cell.


In [7]:
# S7: Deterministic checkpoint record
checkpoint_22 = {
    "mask_row_2": causal_mask[2].int().tolist(),
    "causal_weight_row_0": causal_weights[0].tolist(),
    "attention_weight_shape": tuple(attention_weights.shape),
    "attention_output_shape": tuple(attention_output.shape),
    "head_size": attention.head_size,
    "causal_mask_is_registered_buffer": (
        "causal_mask" in dict(attention.named_buffers())
    ),
    "causal_mask_is_trainable_parameter": (
        "causal_mask" in dict(attention.named_parameters())
    ),
    "prefix_difference": float(prefix_difference.detach()),
    "suffix_changed": bool(suffix_difference.detach() > 1e-5),
}
checkpoint_22


{'mask_row_2': [1, 1, 1, 0],
 'causal_weight_row_0': [1.0, 0.0, 0.0, 0.0],
 'attention_weight_shape': (3, 2, 6, 6),
 'attention_output_shape': (3, 6, 8),
 'head_size': 4,
 'causal_mask_is_registered_buffer': True,
 'causal_mask_is_trainable_parameter': False,
 'prefix_difference': 0.0,
 'suffix_changed': True}

## Pitfall and extension

**Pitfall.** The mask must be applied to the scores *before* softmax. Zeroing attention weights afterward would make the remaining row sum less than one unless it were normalized again.

**Optional extension.** Temporarily remove `masked_fill` from S3 and rerun S5. The prefix-invariance assertion should fail, demonstrating future-token leakage directly.
